# Setup & Data Loading

In [ ]:
!pip install -q catboost xgboost tensorflow scikit-learn pandas numpy

import os
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
# Load data directly from CSV
df = pd.read_csv('training_data_v2.csv')

# Display basic info
print(f"Loaded {len(df)} records with {len(df.columns)} features.")

# Prepare features and target for Failure Prediction
exclude_cols = ['provider_id', 'is_failed', 'reliability_tier']
feature_cols = [c for c in df.columns if c not in exclude_cols]

X = df[feature_cols].copy().fillna(0)
y = df['is_failed'].astype(int)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")

# Scale features (Required for ANN, doesn't hurt trees)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# CART (Decision Tree) Model

In [ ]:
cart_model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=10,
    random_state=42
)
cart_model.fit(X_train, y_train)

cart_pred = cart_model.predict(X_test)

print("CART Accuracy:  {:.2f}%".format(accuracy_score(y_test, cart_pred) * 100))
print("CART Precision: {:.2f}%".format(precision_score(y_test, cart_pred, zero_division=0) * 100))
print("CART Recall:    {:.2f}%".format(recall_score(y_test, cart_pred, zero_division=0) * 100))

# XGBoost Model

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)
xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)

print("XGBoost Accuracy:  {:.2f}%".format(accuracy_score(y_test, xgb_pred) * 100))
print("XGBoost Precision: {:.2f}%".format(precision_score(y_test, xgb_pred, zero_division=0) * 100))
print("XGBoost Recall:    {:.2f}%".format(recall_score(y_test, xgb_pred, zero_division=0) * 100))

# CatBoost Model

In [ ]:
catboost_model = CatBoostClassifier(
    iterations=100,
    depth=5,
    learning_rate=0.1,
    random_seed=42,
    verbose=False
)
catboost_model.fit(X_train, y_train)

cat_pred = catboost_model.predict(X_test)

print("CatBoost Accuracy:  {:.2f}%".format(accuracy_score(y_test, cat_pred) * 100))
print("CatBoost Precision: {:.2f}%".format(precision_score(y_test, cat_pred, zero_division=0) * 100))
print("CatBoost Recall:    {:.2f}%".format(recall_score(y_test, cat_pred, zero_division=0) * 100))

# ANN (Keras) Model

In [ ]:
ann_model = Sequential([
    Dense(32, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

ann_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

ann_model.fit(X_train_scaled, y_train, epochs=200, validation_split=0.1, callbacks=[early_stop], verbose=0)

ann_pred_prob = ann_model.predict(X_test_scaled, verbose=0)
ann_pred = (ann_pred_prob > 0.5).astype(int).flatten()

print("ANN Accuracy:  {:.2f}%".format(accuracy_score(y_test, ann_pred) * 100))
print("ANN Precision: {:.2f}%".format(precision_score(y_test, ann_pred, zero_division=0) * 100))
print("ANN Recall:    {:.2f}%".format(recall_score(y_test, ann_pred, zero_division=0) * 100))